# Lab: Cross-Validation and the Bootstrap

In [1]:
import numpy as np
from ISLP import load_data
from sklearn.model_selection import train_test_split

In [2]:
from functools import partial
from sklearn.model_selection import (cross_validate,
                                     KFold,
                                     ShuffleSplit)
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

## 1. The Validation Set Approach

In [3]:
Auto = load_data('Auto')
auto_train, auto_valid = train_test_split(Auto, test_size=196, random_state=0)

In [4]:
x_train = auto_train[['horsepower']]
y_train = auto_train['mpg']
#insted of following islp package and sm lets try to make it uniform by using sklearn everywhere it is possible
model = make_pipeline(PolynomialFeatures(degree=1, include_bias=False), LinearRegression())
results = model.fit(x_train, y_train)



In [5]:
x_valid = auto_valid[['horsepower']]
y_valid = auto_valid['mpg']
valid_pred = results.predict(x_valid)
np.mean((y_valid - valid_pred)**2)

np.float64(23.61661706966988)

lets create a function which returns MSE for higher-degree polynomial regressions.

In [6]:
def evalMSE(predictors,
            response,
            train,
            test,
            deg):
    '''As predictors and response give the name of columns'''
    X_train = train[[predictors]]
    Y_train = train[response]
    Model = make_pipeline(PolynomialFeatures(degree=deg, include_bias=False), LinearRegression())
    Result = Model.fit(X_train, Y_train)
    X_test = test[[predictors]]
    Y_test = test[response]
    test_pred = Result.predict(X_test)
    return np.mean((Y_test - test_pred)**2)
    
    

In [7]:
auto_train, auto_pred = train_test_split(Auto, test_size=196, random_state=3)
for d in range(1,10):
    MSE = evalMSE('horsepower', 'mpg', auto_train, auto_pred, d)
    print(f'The MSE using a degree {d} is {MSE:.2f}')

The MSE using a degree 1 is 20.76
The MSE using a degree 2 is 16.95
The MSE using a degree 3 is 16.97
The MSE using a degree 4 is 16.90
The MSE using a degree 5 is 16.74
The MSE using a degree 6 is 16.80
The MSE using a degree 7 is 16.81
The MSE using a degree 8 is 16.97
The MSE using a degree 9 is 17.52


## 2. Cross-Validation

In [8]:
#LOOCV 
X = Auto[['horsepower']]
Y = Auto['mpg']
hp_model = make_pipeline(PolynomialFeatures(degree=1, include_bias=False), LinearRegression())
cv_results = cross_validate(hp_model, X, Y, cv=Auto.shape[0], scoring='neg_mean_squared_error')
cv_err = -np.mean(cv_results['test_score'])
cv_err



np.float64(24.231513517929226)

In [9]:
cv_error=[]
for d in range(1,7):
    hp_model_1 = make_pipeline(PolynomialFeatures(degree=d, include_bias=False), LinearRegression())
    cv_results_1 = cross_validate(hp_model_1, X, Y, cv=Auto.shape[0], scoring='neg_mean_squared_error')
    #we had to use neg_mean_squared_error bec by def it had used R2 which measures how much better it is predicting than-
    # mean, while mse measures how wrong are my predition in real unit
    cv_err_1 = -np.mean(cv_results_1['test_score'])
    cv_error.append(cv_err_1)
    
cv_error

[np.float64(24.231513517929226),
 np.float64(19.24821312448967),
 np.float64(19.3349840640292),
 np.float64(19.424430310294134),
 np.float64(19.033213505762642),
 np.float64(19.166486141423878)]

In [10]:
#kfold
cv_error_2 = []
cv = KFold(n_splits=10, shuffle=True, random_state=0)
for d in range(1,6):
    hp_model_2 = make_pipeline(PolynomialFeatures(degree=d, include_bias=False), LinearRegression())
    cv_results_2 = cross_validate(hp_model_2, X, Y, cv=cv, scoring='neg_mean_squared_error')
    cv_err_2 = -np.mean(cv_results_2['test_score'])
    cv_error_2.append(cv_err_2)
cv_error_2
    

[np.float64(24.207664490171332),
 np.float64(19.185331419375096),
 np.float64(19.276266655514625),
 np.float64(19.478484016344122),
 np.float64(19.137203453529324)]

In [11]:
validation = ShuffleSplit(n_splits =1,
                          test_size =196,
                          random_state =0)

results = cross_validate(hp_model,
                            X,Y,
                            cv=validation,
                            scoring='neg_mean_squared_error');
-results['test_score']

array([23.61661707])

In [12]:
validation = ShuffleSplit(n_splits=10,
                          test_size=196,
                          random_state=0)
results = cross_validate(hp_model,
                         X,
                         Y,
                         cv=validation,
                         scoring='neg_mean_squared_error')
-results['test_score'].mean(), results['test_score'].std()

(np.float64(23.80223266103416), np.float64(1.4218450941091856))

## 3. Bootstrap

In [17]:
Portfolio = load_data('Portfolio')
def alpha_func(D, idx):
    cov_ = np.cov(D[['X', 'Y']].loc[idx], rowvar=False)
    return ((cov_[1,1] - cov_[0,1]) / (cov_[0,0] + cov_[1,1] - 2*cov_[0,1]))
    

In [18]:
alpha_func(Portfolio, range(100))

np.float64(0.57583207459283)

In [20]:
rng = np.random.default_rng(0)
alpha_func(Portfolio,
            rng.choice(100,
                       100,
                       replace=True))

np.float64(0.6074452469619004)

In [23]:
def boot_SE(func,
            D,
            n=None,
            B=1000,
            seed=0):
    rng = np.random.default_rng(seed)
    first_, second_ = 0, 0
    n = n or D.shape[0]
    for _ in range(B):
        idx = rng.choice(D.index,
                         n,
                         replace=True)
        value = func(D, idx)
        first_ += value
        second_ +=value**2
    return np.sqrt(second_ / B - (first_ / B)**2)

In [24]:
alpha_SE = boot_SE(alpha_func,
                   Portfolio,
                   B=1000,
                   seed=0)
alpha_SE

np.float64(0.09118176521277699)